# Case 6: Age-related macular degeneration

*Hands-On Electroretinography in the Age of AI*, Chapter 19, §19.6.

Mrs. H. is in the 60y and older age stratum. Her ERG shows one change: a mildly reduced LA 30 Hz flicker amplitude. The aim of this case is to show how the pipeline reports the uncertainty of a low-confidence AMD classification. This notebook assigns the zones of Table 19.7 with the pipeline code, checks zones and printed limits against the book, and shows what a calibrated probability of 0.54 means.

The values are constructed for teaching; they are not patient data. Limits: Baker et al. (2025), silver thread, 60y and older.

**Setup.** In Google Colab, the first cell clones the repository. Elsewhere, run the notebook from the repository.

In [1]:
import os, sys
if 'google.colab' in sys.modules and not os.path.exists('/content/ERG-Analysis-API'):
    !git clone -q https://github.com/AI-Fairness-com/ERG-Analysis-API.git /content/ERG-Analysis-API
sys.path.insert(0, '/content/ERG-Analysis-API/examples')
sys.path.insert(0, os.path.abspath('examples') if os.path.isdir('examples') else os.getcwd())
from case_helpers import zone_table, check, REPORT

**Table 19.7 values.**

In [2]:
ROWS = [
    ('DA 3.0',  'a-wave amplitude',      105,  'GREEN', (80, 220)),
    ('DA 3.0',  'a-wave implicit time',  16.6, 'GREEN', (14.5, 18)),
    ('DA 3.0',  'b-wave amplitude',      195,  'GREEN', (162, 407)),
    ('DA 3.0',  'b-wave implicit time',  53,   'GREEN', (46.5, 58)),
    ('LA 3.0',  'a-wave amplitude',      19,   'GREEN', (12, 35)),
    ('LA 3.0',  'a-wave implicit time',  14.4, 'GREEN', (13, 15.5)),
    ('LA 3.0',  'b-wave amplitude',      52,   'GREEN', (49, 153)),
    ('LA 3.0',  'b-wave implicit time',  29.8, 'GREEN', (27.5, 32.5)),
    ('LA 30 Hz', 'flicker amplitude',    37,   'AMBER', (38, 116)),
    ('LA 30 Hz', 'flicker implicit time', 30.1, 'GREEN', (24.5, 30.5)),
    ('LA 3.0',  'PhNR amplitude',        19.2, 'Not zoned'),
    ('LA 3.0',  'OP2 implicit time',     21.8, 'Not zoned'),
]

**Zones.** The flicker amplitude of 37 µV lies just below the lower limit of 38 µV but above L_lo, so it is AMBER; everything else is GREEN.

In [3]:
table, signal = zone_table(ROWS, age=70)
print(table.drop(columns='Match').to_string(index=False))
print('\nComposite Traffic Light:', signal)
check(table)
flicker = REPORT.classify_parameters({'b_amp_uv': 37}, 'dtl_fiber', 'LA 30 Hz', 70)[0]
print('\nLayer 1 message:', REPORT.composite_traffic_light([flicker], 'dtl_fiber')['message'])

Protocol             Parameter  Value                            Limits Zone (code) Zone (book)
  DA 3.0      a-wave amplitude  105.0     80 to 220 (L_lo 72, U_hi 234)       GREEN       GREEN
  DA 3.0  a-wave implicit time   16.6   14.5 to 18 (L_lo 14, U_hi 18.5)       GREEN       GREEN
  DA 3.0      b-wave amplitude  195.0   162 to 407 (L_lo 151, U_hi 458)       GREEN       GREEN
  DA 3.0  b-wave implicit time   53.0   46.5 to 58 (L_lo 45.5, U_hi 60)       GREEN       GREEN
  LA 3.0      a-wave amplitude   19.0       12 to 35 (L_lo 12, U_hi 41)       GREEN       GREEN
  LA 3.0  a-wave implicit time   14.4 13 to 15.5 (L_lo 12.5, U_hi 16.5)       GREEN       GREEN
  LA 3.0      b-wave amplitude   52.0     49 to 153 (L_lo 45, U_hi 192)       GREEN       GREEN
  LA 3.0  b-wave implicit time   29.8 27.5 to 32.5 (L_lo 26.5, U_hi 33)       GREEN       GREEN
LA 30 Hz     flicker amplitude   37.0     38 to 116 (L_lo 36, U_hi 160)       AMBER       AMBER
LA 30 Hz flicker implicit time   30.1   

**What a calibrated probability of 0.54 means (§19.6.2).** After temperature scaling (Chapter 14 §14.5.2), a probability is a frequency: of every 100 recordings with this score, about 54 come from patients with confirmed AMD and 46 do not. The pipeline reports that probability as it is and does not round it up to sound more decisive. The second-ranked class, early Cone Dystrophy, has a probability of 0.20. The Model Card gives AMD the lowest illustrative AUC of the seven classes (0.81, Table 17.4) and advises heightened specialist review of every non-GREEN AMD classification.

In [4]:
p_amd, p_cone = 0.54, 0.20
print(f'Of 100 recordings with this score: about {round(100 * p_amd)} with AMD, '
      f'{100 - round(100 * p_amd)} without.')
print('Layer 2 Clinical Summary:')
print(f'  The pipeline assigns a probability of {p_amd:.0%} to Age-Related Macular '
      f'Degeneration. This is a low-confidence classification. A similar amplitude '
      f'occurs in some healthy older people and in early Cone Dystrophy (probability '
      f'{p_cone:.0%}). Correlation with the fundus findings and OCT is essential '
      f'before this result is interpreted. Specialist review is recommended.')

Of 100 recordings with this score: about 54 with AMD, 46 without.
Layer 2 Clinical Summary:
  The pipeline assigns a probability of 54% to Age-Related Macular Degeneration. This is a low-confidence classification. A similar amplitude occurs in some healthy older people and in early Cone Dystrophy (probability 20%). Correlation with the fundus findings and OCT is essential before this result is interpreted. Specialist review is recommended.
